In [ ]:
#installing dependencies and libraries
!pip install -q tifffile scikit-image scikit-learn seaborn rasterio

In [ ]:
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tifffile
from pathlib import Path
from PIL import Image, ImageStat
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc, roc_auc_score,ConfusionMatrixDisplay, accuracy_score)
from skimage.feature import graycomatrix, graycoprops
from skimage.transform import resize as sk_resize

directory = Path('PhDMangroveDataset')
mangrove_class = ['mangroves', 'nonmangroves']
labbeled_class = {'mangroves': 1, 'nonmangroves': 0}
zero_value = 0.0
seed = 42
image_shape = (256, 256, 7)

**This section is pre-processing of the mangrove dataset from @Kaggle**

In [ ]:
for cls in mangrove_class:
    folder = directory / cls

def corruption(path: str) -> str:
    try:
        #this is reading the file using tifffile (originally i used the OS method after i couldnt get tifffile to work however i reversed engineered someones kaggle code on this dataset to learn) 
        data = tifffile.imread(path).astype(np.float32)
        if data.ndim == 2:
            data = data[..., np.newaxis] #this is adding a new axis so it goes from 2d to 3d + colour
        elif data.shape[0] < data.shape[-1]:
            data = data.transpose(1, 2, 0) #transposing the data so it goes from (bands, height, width) to (height, width, bands) 256x256x7
        valid = data[data != zero_value]
        if valid.size == 0:           return 'dark' 
        mean = float(valid.mean())
        std  = float(valid.std())
        if mean < 0.01:               return 'dark'
        if std < 0.001:               return 'uniform'
        return 'valid'
    except Exception as e:
        return f'error: {e}'

print(f"{'Class':<20} {'valid':>8} {'dark':>8} {'uniform':>10} {'error':>8}")


#this is going through every image in the dataset and is calling the curruption function to determine if it is valid or not
summary = {}
for cls in mangrove_class:
    folder = directory / cls
    counts = {'valid': 0, 'dark': 0, 'uniform': 0, 'error': 0}
    for tif in folder.glob('*.tif'):
        tag = corruption(str(tif))
        key = tag if tag in counts else 'error'
        counts[key] += 1
    summary[cls] = counts
    print(f"{cls:<20} {counts['valid']:>8,} {counts['dark']:>8,} "
          f"{counts['uniform']:>10,} {counts['error']:>8,}")
# this is printing out a summary of the corruption types for each class in a nice format


def preview(path, size=(128, 128)):
    try:
        data = tifffile.imread(path).astype(np.float32)
        if data.ndim == 2:
            data = np.stack([data] * 3, axis=-1) #same explaination as how we originally read the data
        elif data.shape[0] < data.shape[-1]: 
            data = data.transpose(1, 2, 0) 

        rgb = data[:, :, :3] #taking the first 3 bands as RGB 
        out = np.zeros((*rgb.shape[:2], 3), dtype=np.float32)#creating an empty array to store the normalized RGB values
        for c in range(3):
            lo, hi = np.percentile(rgb[:, :, c], [2, 98])
            if hi > lo:
                out[:, :, c] = np.clip((rgb[:, :, c] - lo) / (hi - lo), 0, 1)
        return sk_resize(out, size, anti_aliasing=True) #resizing the image to the specified size
    except Exception as e:
        return np.zeros((*size, 3))

n_cols = 12
n_rows = 12

for cls in mangrove_class:
    folder  = directory / cls
    tifs    = sorted(folder.glob('*.tif'))
    n_show  = min(n_cols * n_rows, len(tifs))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
    fig.suptitle(f'Class: {cls}  ({len(tifs)} total — showing {n_show})', fontsize=13, y=1.01)

    for i, ax in enumerate(axes.flat): #showing a preview of images in a 12x12
        if i < n_show:
            img  = preview(str(tifs[i]))
            name = tifs[i].stem
            tag  = corruption(str(tifs[i]))
            ax.imshow(img)
            color = 'red' if tag != 'valid' else 'white'
            ax.set_title(f'{name[:10]}\n{tag}', fontsize=6, color=color)
        else:
            ax.axis('off')
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()